# Qwen2.5-0.5B: peS2o Full-Corpus Training-Efficiency Curves

This notebook runs one data variant at a time: `raw`, `minhashlsh`, or `lshbloom`. Unlike the equal 25-million-token experiment, this experiment trains for one epoch over every complete 2,048-token sequence in the selected variant.

During training, every 250 optimizer steps the notebook evaluates the same fixed validation probe and saves a temporary model-only checkpoint. After training, it evaluates the Base model and every checkpoint on all 1,000 SciQ test examples. The output contains quality curves against cumulative training tokens and training-only GPU time.

Select a V100 runtime in Colab and add `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, and `WANDB_API_KEY` to Colab Secrets. Temporary AWS credentials also require `AWS_SESSION_TOKEN`. Change only `VARIANT`, and use a fresh runtime for each of the three variants.


In [ ]:
# Install version-constrained experiment dependencies.
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "transformers>=4.56,<5",
    "accelerate>=1.8,<2",
    "lm-eval[hf]==0.4.13",
    "wandb>=0.21,<1",
    "boto3>=1.35,<2",
])
print("Dependencies installed. Restart the runtime only if Colab asks for it.")


In [ ]:
from __future__ import annotations

import gzip
import hashlib
import json
import os
import tempfile
from pathlib import Path
from typing import Iterator

import numpy as np


def _iter_records(path: Path) -> Iterator[dict]:
    with gzip.open(path, "rt", encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"invalid JSON at line {line_number} in {path}") from error
            if not isinstance(record, dict):
                raise ValueError(f"record at line {line_number} must be an object")
            for field in ("id", "text"):
                if not isinstance(record.get(field), str) or not record[field]:
                    raise ValueError(
                        f"record field {field!r} at line {line_number} must be non-empty"
                    )
            yield record


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def pack_jsonl_gz_to_memmap(
    input_path: Path | str,
    output_path: Path | str,
    tokenizer: object,
    sequence_length: int,
    sequence_count: int,
) -> dict:
    input_path = Path(input_path).resolve()
    output_path = Path(output_path).resolve()
    if not input_path.is_file():
        raise FileNotFoundError(input_path)
    if output_path.exists():
        raise FileExistsError(output_path)
    if sequence_length < 2:
        raise ValueError("sequence_length must be at least 2")
    if sequence_count < 1:
        raise ValueError("sequence_count must be positive")
    eos_token_id = getattr(tokenizer, "eos_token_id", None)
    if eos_token_id is None:
        raise ValueError("tokenizer must define eos_token_id")

    required_tokens = sequence_length * sequence_count
    output_path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(
        prefix=f".{output_path.name}.", dir=output_path.parent
    )
    os.close(descriptor)
    temporary_path = Path(temporary_name)
    tokens_written = 0
    documents_read = 0
    last_document_id = None
    last_document_tokens_used = 0
    last_document_complete = False

    try:
        packed = np.memmap(
            temporary_path, mode="w+", dtype=np.uint32, shape=(required_tokens,)
        )
        for record in _iter_records(input_path):
            token_ids = tokenizer.encode(record["text"], add_special_tokens=False)
            token_ids.append(eos_token_id)
            if token_ids and (min(token_ids) < 0 or max(token_ids) > np.iinfo(np.uint32).max):
                raise ValueError(f"token id outside uint32 range in document {record['id']}")

            documents_read += 1
            last_document_id = record["id"]
            remaining = required_tokens - tokens_written
            last_document_tokens_used = min(remaining, len(token_ids))
            end = tokens_written + last_document_tokens_used
            packed[tokens_written:end] = token_ids[:last_document_tokens_used]
            tokens_written = end
            last_document_complete = last_document_tokens_used == len(token_ids)
            if tokens_written == required_tokens:
                break

        packed.flush()
        del packed
        if tokens_written != required_tokens:
            raise ValueError(
                f"packing requires {required_tokens} tokens, found {tokens_written}"
            )
        os.replace(temporary_path, output_path)
    except BaseException:
        temporary_path.unlink(missing_ok=True)
        raise

    return {
        "input_path": str(input_path),
        "output_path": str(output_path),
        "dtype": "uint32",
        "sequence_length": sequence_length,
        "sequence_count": sequence_count,
        "input_tokens": required_tokens,
        "documents_read": documents_read,
        "last_document_id": last_document_id,
        "last_document_tokens_used": last_document_tokens_used,
        "last_document_complete": last_document_complete,
        "tokenizer": getattr(tokenizer, "name_or_path", type(tokenizer).__name__),
        "sha256": _sha256(output_path),
        "bytes": output_path.stat().st_size,
    }


class TokenMemmapDataset:
    def __init__(
        self, path: Path | str, sequence_length: int, sequence_count: int
    ) -> None:
        self.path = Path(path).resolve()
        self.sequence_length = sequence_length
        self.sequence_count = sequence_count
        expected_bytes = sequence_length * sequence_count * np.dtype(np.uint32).itemsize
        if self.path.stat().st_size != expected_bytes:
            raise ValueError(
                f"memmap size is {self.path.stat().st_size} bytes, expected {expected_bytes}"
            )
        self._tokens = np.memmap(
            self.path,
            mode="r",
            dtype=np.uint32,
            shape=(sequence_count, sequence_length),
        )

    def __len__(self) -> int:
        return self.sequence_count

    def __getitem__(self, index: int) -> dict:
        if index < 0:
            index += self.sequence_count
        if index < 0 or index >= self.sequence_count:
            raise IndexError(index)
        return {"input_ids": self._tokens[index].astype(np.int64)}


In [ ]:
from __future__ import annotations

import gzip
import io
import json
import math
import time
import urllib.request
from pathlib import Path
from typing import Any, Callable, Iterable, Iterator


SUPPORTED_SOURCES = {"s2orc", "s2ag"}


def source_family(source: str) -> str:
    family = source.split("/", maxsplit=1)[0]
    if family not in SUPPORTED_SOURCES:
        raise ValueError(f"unsupported source {source}")
    return family


def validate_record(record: dict, expected_source: str | None = None) -> dict:
    if not isinstance(record, dict):
        raise ValueError("each record must be a JSON object")
    for key in ("id", "source", "text"):
        if not isinstance(record.get(key), str) or not record[key]:
            raise ValueError(f"record field {key!r} must be a non-empty string")
    record_source = source_family(record["source"])
    if expected_source is not None and record_source != source_family(expected_source):
        raise ValueError(
            f"expected source {expected_source}, received {record['source']}"
        )
    return record


def iter_jsonl_gz(url: str) -> Iterator[dict]:
    with urllib.request.urlopen(url, timeout=300) as response:
        with gzip.GzipFile(fileobj=response) as compressed:
            with io.TextIOWrapper(compressed, encoding="utf-8") as text_stream:
                for line_number, line in enumerate(text_stream, start=1):
                    if not line.strip():
                        continue
                    try:
                        record = json.loads(line)
                    except json.JSONDecodeError as error:
                        raise ValueError(
                            f"invalid JSON at line {line_number} from {url}"
                        ) from error
                    yield validate_record(record)


def collect_source_records(
    urls: Iterable[str], limits: dict[str, int | None]
) -> dict[str, list[dict]]:
    unknown_sources = set(limits) - SUPPORTED_SOURCES
    if unknown_sources:
        raise ValueError(f"unsupported requested sources: {sorted(unknown_sources)}")
    for source, limit in limits.items():
        if limit is not None and limit < 1:
            raise ValueError(f"limit for {source} must be positive or None")

    records = {source: [] for source in limits}

    def all_finite_limits_reached() -> bool:
        return all(
            limit is not None and len(records[source]) >= limit
            for source, limit in limits.items()
        )

    for url in urls:
        for record in iter_jsonl_gz(url):
            source = source_family(record["source"])
            if source not in records:
                continue
            limit = limits[source]
            if limit is None or len(records[source]) < limit:
                records[source].append(record)
            if all_finite_limits_reached():
                return records

    short = {
        source: {"expected": limit, "actual": len(records[source])}
        for source, limit in limits.items()
        if limit is not None and len(records[source]) < limit
    }
    if short:
        raise ValueError(f"validation streams ended before limits were met: {short}")
    return records


def iter_packed_sequences(
    records: Iterable[dict], tokenizer: Any, sequence_length: int
) -> Iterator[dict]:
    if sequence_length < 2:
        raise ValueError("sequence_length must be at least 2")
    eos_token_id = tokenizer.eos_token_id
    if eos_token_id is None:
        raise ValueError("tokenizer must define eos_token_id")

    token_buffer: list[int] = []
    origin_buffer: list[str] = []

    for raw_record in records:
        record = validate_record(raw_record)
        document_id = record["id"]
        token_ids = tokenizer.encode(record["text"], add_special_tokens=False)
        token_ids.append(eos_token_id)

        position = 0
        while position < len(token_ids):
            space = sequence_length - len(token_buffer)
            next_position = min(position + space, len(token_ids))
            piece = token_ids[position:next_position]
            token_buffer.extend(piece)
            origin_buffer.extend([document_id] * len(piece))
            position = next_position

            if len(token_buffer) == sequence_length:
                yield {
                    "input_ids": token_buffer,
                    "labels": token_buffer.copy(),
                    "document_ids": list(dict.fromkeys(origin_buffer)),
                }
                token_buffer = []
                origin_buffer = []

    if token_buffer:
        padding = sequence_length - len(token_buffer)
        yield {
            "input_ids": token_buffer + [eos_token_id] * padding,
            "labels": token_buffer + [-100] * padding,
            "document_ids": list(dict.fromkeys(origin_buffer)),
        }


def perplexity_from_loss(mean_loss: float) -> float:
    if not math.isfinite(mean_loss):
        raise ValueError("mean loss must be finite")
    try:
        return math.exp(mean_loss)
    except OverflowError:
        return float("inf")


def combine_source_metrics(metrics: Iterable[dict]) -> dict:
    metrics = list(metrics)
    total_tokens = sum(int(item["predicted_tokens"]) for item in metrics)
    if total_tokens <= 0:
        raise ValueError("predicted token total must be positive")
    total_nll = sum(float(item["negative_log_likelihood"]) for item in metrics)
    loss = total_nll / total_tokens
    return {
        "negative_log_likelihood": total_nll,
        "predicted_tokens": total_tokens,
        "loss": loss,
        "perplexity": perplexity_from_loss(loss),
    }


def count_shifted_targets(labels: Any) -> int:
    return int(labels[:, 1:].ne(-100).sum().item())


def _iter_batches(items: Iterable[dict], batch_size: int) -> Iterator[list[dict]]:
    if batch_size < 1:
        raise ValueError("batch_size must be positive")
    batch: list[dict] = []
    for item in items:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


def evaluate_source(
    model: Any,
    packed_sequences: Iterable[dict],
    source: str,
    batch_size: int,
    device: str,
    log_every_steps: int,
    progress_callback: Callable[[dict], None] | None = None,
) -> dict:
    import torch

    if source not in SUPPORTED_SOURCES:
        raise ValueError(f"unsupported source {source}")
    if log_every_steps < 1:
        raise ValueError("log_every_steps must be positive")

    start = time.perf_counter()
    total_nll = 0.0
    predicted_tokens = 0
    input_tokens = 0
    sequence_count = 0
    batch_count = 0
    document_ids: list[str] = []
    seen_document_ids: set[str] = set()
    last_logged_batch = 0

    try:
        with torch.inference_mode():
            for batch_count, batch in enumerate(
                _iter_batches(packed_sequences, batch_size), start=1
            ):
                input_ids = torch.tensor(
                    [item["input_ids"] for item in batch],
                    dtype=torch.long,
                    device=device,
                )
                labels = torch.tensor(
                    [item["labels"] for item in batch],
                    dtype=torch.long,
                    device=device,
                )
                attention_mask = labels.ne(-100).long()
                batch_targets = count_shifted_targets(labels)
                if batch_targets < 1:
                    raise ValueError("a packed batch contained no prediction targets")

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )
                batch_loss = float(outputs.loss.item())
                if not math.isfinite(batch_loss):
                    raise ValueError(
                        f"non-finite loss for {source} at batch {batch_count}"
                    )

                total_nll += batch_loss * batch_targets
                predicted_tokens += batch_targets
                input_tokens += int(labels.ne(-100).sum().item())
                sequence_count += len(batch)
                for item in batch:
                    for document_id in item["document_ids"]:
                        if document_id not in seen_document_ids:
                            seen_document_ids.add(document_id)
                            document_ids.append(document_id)

                if progress_callback is not None and (
                    batch_count == 1 or batch_count % log_every_steps == 0
                ):
                    elapsed = max(time.perf_counter() - start, 1e-9)
                    running_loss = total_nll / predicted_tokens
                    progress_callback(
                        {
                            "source": source,
                            "batch": batch_count,
                            "sequences": sequence_count,
                            "predicted_tokens": predicted_tokens,
                            "loss": running_loss,
                            "perplexity": perplexity_from_loss(running_loss),
                            "tokens_per_second": predicted_tokens / elapsed,
                            "elapsed_seconds": elapsed,
                            "gpu_memory_gb": torch.cuda.memory_allocated() / (1024**3),
                        }
                    )
                    last_logged_batch = batch_count
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError(
            f"CUDA ran out of memory with batch_size={batch_size}; "
            "reduce CONFIG['batch_size'] and rerun"
        ) from error

    if predicted_tokens < 1:
        raise ValueError(f"no prediction targets were evaluated for {source}")

    elapsed = max(time.perf_counter() - start, 1e-9)
    loss = total_nll / predicted_tokens
    result = {
        "source": source,
        "documents": len(document_ids),
        "document_ids": document_ids,
        "sequences": sequence_count,
        "batches": batch_count,
        "input_tokens": input_tokens,
        "predicted_tokens": predicted_tokens,
        "negative_log_likelihood": total_nll,
        "loss": loss,
        "perplexity": perplexity_from_loss(loss),
        "elapsed_seconds": elapsed,
        "tokens_per_second": predicted_tokens / elapsed,
    }

    if progress_callback is not None and last_logged_batch != batch_count:
        progress_callback({**result, "batch": batch_count})
    return result


def write_result_json(path: str | Path, result: dict) -> Path:
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        json.dumps(result, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False)
        + "\n",
        encoding="utf-8",
    )
    return output_path


In [ ]:
from __future__ import annotations

import json
import math
from collections.abc import Mapping
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any


VARIANTS = ("raw", "minhashlsh", "lshbloom")
STAGE_SAMPLE_COUNTS = {"smoke": 10, "full": 1000}
METRIC_KEYS = {
    "acc": "acc,none",
    "acc_stderr": "acc_stderr,none",
    "acc_norm": "acc_norm,none",
    "acc_norm_stderr": "acc_norm_stderr,none",
}


def _finite_float(value: object, name: str) -> float:
    try:
        converted = float(value)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{name} must be a number") from error
    if not math.isfinite(converted):
        raise ValueError(f"{name} must be finite")
    return converted


def extract_sciq_metrics(
    result: dict,
    variant: str,
    stage: str,
    elapsed_seconds: float,
    peak_cuda_bytes: int,
) -> dict:
    if variant not in VARIANTS:
        raise ValueError(f"unsupported variant: {variant}")
    if stage not in STAGE_SAMPLE_COUNTS:
        raise ValueError(f"unsupported stage: {stage}")

    try:
        metrics = result["results"]["sciq"]
        effective = int(result["n-samples"]["sciq"]["effective"])
    except (KeyError, TypeError, ValueError) as error:
        raise ValueError("invalid SciQ harness result structure") from error

    expected = STAGE_SAMPLE_COUNTS[stage]
    if effective != expected:
        raise ValueError(
            f"{stage} SciQ evaluation expected {expected} examples, received {effective}"
        )

    extracted = {}
    for output_name, harness_name in METRIC_KEYS.items():
        if harness_name not in metrics:
            raise ValueError(f"missing SciQ metric: {harness_name}")
        extracted[output_name] = _finite_float(metrics[harness_name], harness_name)

    elapsed = _finite_float(elapsed_seconds, "elapsed_seconds")
    if elapsed <= 0:
        raise ValueError("elapsed_seconds must be positive")
    try:
        peak_memory = int(peak_cuda_bytes)
    except (TypeError, ValueError) as error:
        raise ValueError("peak_cuda_bytes must be an integer") from error
    if peak_memory < 0:
        raise ValueError("peak_cuda_bytes must be non-negative")

    return {
        "variant": variant,
        "stage": stage,
        "examples": effective,
        **extracted,
        "elapsed_seconds": elapsed,
        "peak_cuda_bytes": peak_memory,
    }


def build_comparison(rows: list[dict]) -> list[dict]:
    indexed = {}
    for row in rows:
        variant = row.get("variant")
        if variant in indexed:
            raise ValueError(f"duplicate variant: {variant}")
        indexed[variant] = row

    missing = [variant for variant in VARIANTS if variant not in indexed]
    if missing:
        raise ValueError(f"missing variants: {', '.join(missing)}")

    for variant in VARIANTS:
        if indexed[variant].get("stage") != "full":
            raise ValueError("comparison requires full-stage rows")

    raw = indexed["raw"]
    comparison = []
    for variant in VARIANTS:
        row = dict(indexed[variant])
        row["delta_acc"] = float(row["acc"]) - float(raw["acc"])
        row["delta_acc_norm"] = float(row["acc_norm"]) - float(raw["acc_norm"])
        comparison.append(row)
    return comparison


def to_jsonable(value: object) -> object:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if is_dataclass(value) and not isinstance(value, type):
        return to_jsonable(asdict(value))
    if isinstance(value, Mapping):
        return {str(key): to_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(item) for item in value]
    if isinstance(value, set):
        return [to_jsonable(item) for item in sorted(value, key=repr)]

    item = getattr(value, "item", None)
    if callable(item):
        converted = item()
        if converted is not value:
            return to_jsonable(converted)

    tolist = getattr(value, "tolist", None)
    if callable(tolist):
        converted = tolist()
        if converted is not value:
            return to_jsonable(converted)

    return str(value)


def write_json(path: str | Path, value: Any) -> Path:
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        json.dumps(
            to_jsonable(value),
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )
    return output_path


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
from collections.abc import Iterable, Mapping
from pathlib import Path


VARIANTS = ("raw", "minhashlsh", "lshbloom")
CURVE_COLUMNS = (
    "variant",
    "global_step",
    "cumulative_train_tokens",
    "cumulative_train_gpu_hours",
    "validation_loss",
    "validation_perplexity",
    "sciq_acc",
    "sciq_acc_norm",
    "sciq_acc_stderr",
    "sciq_acc_norm_stderr",
    "sciq_examples",
)


def canonical_sha256(value: object) -> str:
    """Return a stable SHA-256 for JSON-compatible experiment metadata."""
    try:
        encoded = json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        ).encode("utf-8")
    except (TypeError, ValueError) as error:
        raise ValueError("value must contain only finite JSON-compatible data") from error
    return hashlib.sha256(encoded).hexdigest()


def validate_shared_experiment_results(
    results: Iterable[Mapping[str, object]],
    expected_variants: Iterable[str] = VARIANTS,
) -> str:
    """Verify that separately produced variant results belong to one experiment."""
    results = [dict(result) for result in results]
    expected_variants = tuple(expected_variants)
    if len(results) != len(expected_variants):
        raise ValueError("one experiment result is required for every variant")

    variants = [result.get("variant") for result in results]
    if set(variants) != set(expected_variants) or len(set(variants)) != len(variants):
        raise ValueError("experiment result variants do not match the expected variants")

    verified = []
    for result in results:
        if result.get("schema_version") != 2:
            raise ValueError("experiment results require schema_version 2")
        identity = result.get("experiment_identity")
        fingerprint = result.get("experiment_fingerprint")
        if not isinstance(identity, Mapping) or not isinstance(fingerprint, str):
            raise ValueError("experiment identity and fingerprint are required")
        computed = canonical_sha256(identity)
        if computed != fingerprint:
            raise ValueError(
                f"experiment fingerprint is invalid for variant {result.get('variant')}"
            )
        verified.append(fingerprint)

    if len(set(verified)) != 1:
        raise ValueError("experiment fingerprints differ across variants")
    return verified[0]


def _positive_integer(value: object, name: str) -> int:
    if isinstance(value, bool):
        raise ValueError(f"{name} must be a positive integer")
    try:
        converted = int(value)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{name} must be a positive integer") from error
    if converted < 1 or converted != value:
        raise ValueError(f"{name} must be a positive integer")
    return converted


def _finite_number(value: object, name: str) -> float:
    try:
        converted = float(value)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{name} must be a finite number") from error
    if not math.isfinite(converted):
        raise ValueError(f"{name} must be a finite number")
    return converted


def build_full_training_plan(
    token_counts: Mapping[str, int], sequence_length: int
) -> dict[str, dict[str, int]]:
    sequence_length = _positive_integer(sequence_length, "sequence_length")
    if not token_counts:
        raise ValueError("token_counts must not be empty")

    plan = {}
    for variant, raw_count in token_counts.items():
        available_tokens = _positive_integer(
            raw_count, f"available tokens for {variant}"
        )
        sequence_count = available_tokens // sequence_length
        if sequence_count < 1:
            raise ValueError(
                f"{variant} has no complete sequence of length {sequence_length}"
            )
        train_input_tokens = sequence_count * sequence_length
        plan[str(variant)] = {
            "available_tokens": available_tokens,
            "sequence_count": sequence_count,
            "train_input_tokens": train_input_tokens,
            "unused_tail_tokens": available_tokens - train_input_tokens,
        }
    return plan


def checkpoint_steps(total_optimizer_steps: int, save_steps: int) -> list[int]:
    total_optimizer_steps = _positive_integer(
        total_optimizer_steps, "total_optimizer_steps"
    )
    save_steps = _positive_integer(save_steps, "save_steps")
    steps = [0]
    steps.extend(range(save_steps, total_optimizer_steps + 1, save_steps))
    if steps[-1] != total_optimizer_steps:
        steps.append(total_optimizer_steps)
    return steps


def tokens_at_step(
    global_step: int,
    tokens_per_optimizer_step: int,
    total_train_tokens: int,
) -> int:
    if isinstance(global_step, bool):
        raise ValueError("global_step must be a non-negative integer")
    try:
        global_step = int(global_step)
    except (TypeError, ValueError) as error:
        raise ValueError("global_step must be a non-negative integer") from error
    if global_step < 0:
        raise ValueError("global_step must be a non-negative integer")
    tokens_per_optimizer_step = _positive_integer(
        tokens_per_optimizer_step, "tokens_per_optimizer_step"
    )
    total_train_tokens = _positive_integer(total_train_tokens, "total_train_tokens")
    return min(global_step * tokens_per_optimizer_step, total_train_tokens)


def _index_measurements(rows: Iterable[dict], label: str) -> dict[tuple[str, int], dict]:
    indexed = {}
    for row in rows:
        if not isinstance(row, dict):
            raise ValueError(f"each {label} measurement must be an object")
        variant = row.get("variant")
        if not isinstance(variant, str) or not variant:
            raise ValueError(f"each {label} measurement requires variant")
        step = row.get("global_step")
        if isinstance(step, bool):
            raise ValueError(f"each {label} measurement requires integer global_step")
        try:
            step = int(step)
        except (TypeError, ValueError) as error:
            raise ValueError(
                f"each {label} measurement requires integer global_step"
            ) from error
        if step < 0:
            raise ValueError(f"each {label} measurement requires non-negative global_step")
        key = (variant, step)
        if key in indexed:
            raise ValueError(f"duplicate {label} measurement: {variant} step {step}")
        indexed[key] = dict(row, global_step=step)
    return indexed


def merge_curve_measurements(
    validation_rows: Iterable[dict], sciq_rows: Iterable[dict]
) -> list[dict]:
    validation = _index_measurements(validation_rows, "validation")
    sciq = _index_measurements(sciq_rows, "SciQ")
    if validation.keys() != sciq.keys():
        missing_sciq = sorted(validation.keys() - sciq.keys())
        missing_validation = sorted(sciq.keys() - validation.keys())
        raise ValueError(
            "validation and SciQ measurement keys differ: "
            f"missing SciQ={missing_sciq}, missing validation={missing_validation}"
        )

    merged = []
    for key in sorted(validation, key=lambda item: (VARIANTS.index(item[0]) if item[0] in VARIANTS else len(VARIANTS), item[0], item[1])):
        row = dict(validation[key])
        for field, value in sciq[key].items():
            if field in ("variant", "global_step"):
                continue
            if field in row and row[field] != value:
                raise ValueError(f"conflicting field {field!r} for {key}")
            row[field] = value
        merged.append(row)

    expected_variants = tuple(dict.fromkeys(row["variant"] for row in merged))
    validate_curve_rows(merged, expected_variants=expected_variants)
    return merged


def validate_curve_rows(
    rows: Iterable[dict], expected_variants: Iterable[str] = VARIANTS
) -> list[dict]:
    rows = [dict(row) for row in rows]
    expected_variants = tuple(expected_variants)
    if not rows:
        raise ValueError("curve rows must not be empty")

    by_variant = {variant: [] for variant in expected_variants}
    for row in rows:
        missing = [column for column in CURVE_COLUMNS if column not in row]
        if missing:
            raise ValueError(f"curve row is missing columns: {', '.join(missing)}")
        variant = row["variant"]
        if variant not in by_variant:
            raise ValueError(f"unexpected variant: {variant}")
        by_variant[variant].append(row)

        for field in (
            "global_step",
            "cumulative_train_tokens",
            "sciq_examples",
        ):
            value = _finite_number(row[field], field)
            if value < 0 or int(value) != value:
                raise ValueError(f"{field} must be a non-negative integer")
        for field in (
            "cumulative_train_gpu_hours",
            "validation_loss",
            "validation_perplexity",
            "sciq_acc",
            "sciq_acc_norm",
            "sciq_acc_stderr",
            "sciq_acc_norm_stderr",
        ):
            value = _finite_number(row[field], field)
            if value < 0:
                raise ValueError(f"{field} must be non-negative")
        if not 0 <= float(row["sciq_acc"]) <= 1:
            raise ValueError("sciq_acc must be between zero and one")
        if not 0 <= float(row["sciq_acc_norm"]) <= 1:
            raise ValueError("sciq_acc_norm must be between zero and one")

    for variant, variant_rows in by_variant.items():
        if not variant_rows:
            raise ValueError(f"missing variant: {variant}")
        variant_rows.sort(key=lambda row: int(row["global_step"]))
        if int(variant_rows[0]["global_step"]) != 0:
            raise ValueError(f"{variant} curve must begin at global step zero")
        for previous, current in zip(variant_rows, variant_rows[1:]):
            if int(current["global_step"]) <= int(previous["global_step"]):
                raise ValueError(f"{variant} global steps must strictly increase")
            if int(current["cumulative_train_tokens"]) <= int(
                previous["cumulative_train_tokens"]
            ):
                raise ValueError(f"{variant} token counts must strictly increase")
            if float(current["cumulative_train_gpu_hours"]) <= float(
                previous["cumulative_train_gpu_hours"]
            ):
                raise ValueError(f"{variant} GPU hours must strictly increase")
    return rows


def write_curve_csv(path: str | Path, rows: Iterable[dict]) -> Path:
    rows = [dict(row) for row in rows]
    if not rows:
        raise ValueError("curve rows must not be empty")
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=CURVE_COLUMNS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    return output_path


## 1. Configuration and credentials

The only normal edit is `VARIANT`. The validation probe contains 128 fixed sequences. It is used for the curve only; the final checkpoint also receives the full 1,000-document peS2o validation evaluation.


In [ ]:
import gc
import hashlib
import importlib.metadata
import itertools
import json
import math
import os
import platform
import random
import shutil
import sys
import time
from pathlib import Path

import boto3
import numpy as np
import torch
import transformers
import wandb
from google.colab import userdata
from lm_eval import simple_evaluate
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)


VARIANT = "raw"
ALLOWED_VARIANTS = set(VARIANTS)

CONFIG = {
    "model_id": "Qwen/Qwen2.5-0.5B",
    "model_revision": "060db6499f32faf8b98477b0a26969ef7d8b9987",
    "required_gpu_substring": "V100",
    "sequence_length": 2048,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 5e-5,
    "warmup_steps": 50,
    "weight_decay": 0.1,
    "max_grad_norm": 1.0,
    "fp16": True,
    "seed": 42,
    "curve_interval_steps": 250,
    "logging_steps": 10,
    "curve_validation_sequences": {"s2orc": 64, "s2ag": 64},
    "full_validation_documents": {"s2orc": 320, "s2ag": 680},
    "eval_batch_size": 4,
    "sciq_batch_size": 8,
    "sciq_smoke_examples": 10,
    "sciq_examples": 1000,
    "sciq_bootstrap_iters": 1000,
    "wandb_project": "lshbloom-pes2o",
    "wandb_group": "qwen2.5-0.5b-pes2o-dedup-full-efficiency",
}
SCIQ_REVISION = "2c94ad3e1aafab77146f384e23536f97a4849815"
SCIQ_TASK = {
    "task": "sciq",
    "dataset_path": "allenai/sciq",
    "dataset_name": None,
    "dataset_kwargs": {"revision": SCIQ_REVISION},
    "output_type": "multiple_choice",
    "training_split": "train",
    "validation_split": "validation",
    "test_split": "test",
    "doc_to_text": "{{support.lstrip()}}\nQuestion: {{question}}\nAnswer:",
    "doc_to_target": 3,
    "doc_to_choice": "{{[distractor1, distractor2, distractor3, correct_answer]}}",
    "should_decontaminate": True,
    "doc_to_decontamination_query": "{{support}} {{question}}",
    "metric_list": [
        {"metric": "acc", "aggregation": "mean", "higher_is_better": True},
        {"metric": "acc_norm", "aggregation": "mean", "higher_is_better": True},
    ],
    "metadata": {"version": 1.0},
}

if VARIANT not in ALLOWED_VARIANTS:
    raise ValueError(f"VARIANT must be one of {sorted(ALLOWED_VARIANTS)}")
if sum(CONFIG["curve_validation_sequences"].values()) != 128:
    raise RuntimeError("Curve validation probe must contain exactly 128 sequences")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a GPU runtime in Colab.")
gpu_name = torch.cuda.get_device_name(0)
if CONFIG["required_gpu_substring"] not in gpu_name:
    raise RuntimeError(
        f"This experiment requires a comparable V100 runtime; received {gpu_name}"
    )

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])


def optional_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


try:
    AWS_ACCESS_KEY_ID = userdata.get("AWS_ACCESS_KEY_ID")
    AWS_SECRET_ACCESS_KEY = userdata.get("AWS_SECRET_ACCESS_KEY")
    WANDB_API_KEY = userdata.get("WANDB_API_KEY")
except Exception as error:
    raise RuntimeError("One or more required Colab secrets are unavailable") from error

for name, value in (
    ("AWS_ACCESS_KEY_ID", AWS_ACCESS_KEY_ID),
    ("AWS_SECRET_ACCESS_KEY", AWS_SECRET_ACCESS_KEY),
    ("WANDB_API_KEY", WANDB_API_KEY),
):
    if not value:
        raise RuntimeError(f"Missing required Colab secret: {name}")

AWS_SESSION_TOKEN = optional_secret("AWS_SESSION_TOKEN")
AWS_DEFAULT_REGION = optional_secret("AWS_DEFAULT_REGION") or "ap-northeast-1"
session_kwargs = {
    "aws_access_key_id": AWS_ACCESS_KEY_ID,
    "aws_secret_access_key": AWS_SECRET_ACCESS_KEY,
    "region_name": AWS_DEFAULT_REGION,
}
if AWS_SESSION_TOKEN:
    session_kwargs["aws_session_token"] = AWS_SESSION_TOKEN
s3 = boto3.session.Session(**session_kwargs).client("s3")

S3_BUCKET = "calista-bucket"
S3_SOURCE_PREFIX = "pes2o/v2/experiments/pilot-5000/"
S3_OUTPUT_PREFIX = f"{S3_SOURCE_PREFIX}efficiency/{VARIANT}/"
WORK_DIR = Path(f"/content/pes2o-efficiency-{VARIANT}")
DATA_PATH = WORK_DIR / "train.jsonl.gz"
MANIFEST_PATH = WORK_DIR / "manifest.json"
MEMMAP_PATH = WORK_DIR / "train-tokens.uint32"
TRAINER_DIR = WORK_DIR / "trainer"
FINAL_MODEL_DIR = WORK_DIR / "final"
RESULTS_DIR = WORK_DIR / "results"
CURVE_RESULT_PATH = RESULTS_DIR / "curve-results.json"
CURVE_CSV_PATH = RESULTS_DIR / "curve-results.csv"
TRAINING_PROGRESS_PATH = RESULTS_DIR / "training-progress.json"
SCIQ_PROGRESS_PATH = RESULTS_DIR / "sciq-progress.json"
WORK_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("GPU:", gpu_name)
print("Variant:", VARIANT)
print("S3 output:", f"s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}")


## 2. Download the selected dataset and use its full token budget

Only the incomplete tail shorter than 2,048 tokens is discarded. This is at most 2,047 tokens and is reported explicitly.


In [ ]:
s3.download_file(S3_BUCKET, f"{S3_SOURCE_PREFIX}manifest.json", str(MANIFEST_PATH))
source_manifest_sha256 = _sha256(MANIFEST_PATH)
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
token_counts = {
    name: int(manifest["variants"][name]["token_count"])
    for name in VARIANTS
}
training_plan = build_full_training_plan(token_counts, CONFIG["sequence_length"])
variant_plan = training_plan[VARIANT]
variant_info = manifest["variants"][VARIANT]

s3.download_file(
    S3_BUCKET,
    f"{S3_SOURCE_PREFIX}{variant_info['relative_path']}",
    str(DATA_PATH),
)
if _sha256(DATA_PATH) != variant_info["sha256"]:
    raise RuntimeError("Dataset SHA-256 mismatch")

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_id"], revision=CONFIG["model_revision"], use_fast=True
)
packing = pack_jsonl_gz_to_memmap(
    input_path=DATA_PATH,
    output_path=MEMMAP_PATH,
    tokenizer=tokenizer,
    sequence_length=CONFIG["sequence_length"],
    sequence_count=variant_plan["sequence_count"],
)
train_dataset = TokenMemmapDataset(
    MEMMAP_PATH,
    sequence_length=CONFIG["sequence_length"],
    sequence_count=variant_plan["sequence_count"],
)

updates_per_epoch = math.ceil(
    len(train_dataset)
    / (CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"])
)
tokens_per_update = (
    CONFIG["sequence_length"]
    * CONFIG["per_device_train_batch_size"]
    * CONFIG["gradient_accumulation_steps"]
)
expected_steps = checkpoint_steps(updates_per_epoch, CONFIG["curve_interval_steps"])

print(json.dumps(variant_plan, indent=2))
print("Optimizer updates:", updates_per_epoch)
print("Curve steps:", expected_steps)


## 3. Build a small fixed validation probe

The same document order and sequence counts are used in every variant run. Perplexity at intermediate checkpoints is therefore directly comparable.


In [ ]:
VALIDATION_REVISION = "636a503e44a3ca1b58e01fb61eab0825cd574de0"
VALIDATION_URLS = [
    f"https://huggingface.co/datasets/allenai/peS2o/resolve/{VALIDATION_REVISION}/data/v2/validation-00000-of-00002.json.gz",
    f"https://huggingface.co/datasets/allenai/peS2o/resolve/{VALIDATION_REVISION}/data/v2/validation-00001-of-00002.json.gz",
]
validation_records = collect_source_records(
    VALIDATION_URLS,
    CONFIG["full_validation_documents"],
)


class PackedListDataset:
    def __init__(self, items):
        self.items = list(items)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]


probe_sequences = []
probe_sequence_hashes = {}
for source in ("s2orc", "s2ag"):
    requested = CONFIG["curve_validation_sequences"][source]
    items = list(
        itertools.islice(
            iter_packed_sequences(
                validation_records[source], tokenizer, CONFIG["sequence_length"]
            ),
            requested,
        )
    )
    if len(items) != requested:
        raise RuntimeError(
            f"Validation probe has only {len(items)} {source} sequences; need {requested}"
        )
    probe_sequences.extend(items)
    probe_sequence_hashes[source] = [
        hashlib.sha256(
            np.asarray(item["input_ids"], dtype="<u4").tobytes(order="C")
        ).hexdigest()
        for item in items
    ]
curve_eval_dataset = PackedListDataset(probe_sequences)
probe_identity = {
    "dataset": "allenai/peS2o",
    "revision": VALIDATION_REVISION,
    "sequence_token_sha256": probe_sequence_hashes,
}
probe_sha256 = canonical_sha256(probe_identity)

environment_compatibility = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "lm_eval": importlib.metadata.version("lm-eval"),
    "gpu": gpu_name,
}
experiment_identity = {
    "schema_version": 2,
    "config": CONFIG,
    "training_plan": training_plan,
    "source_manifest_sha256": source_manifest_sha256,
    "sciq_task": SCIQ_TASK,
    "validation_probe": {
        **probe_identity,
        "probe_sha256": probe_sha256,
    },
    "environment": environment_compatibility,
}
experiment_fingerprint = canonical_sha256(experiment_identity)


def causal_lm_collator(features):
    input_ids = torch.from_numpy(
        np.stack([feature["input_ids"] for feature in features])
    ).long()
    labels = torch.from_numpy(
        np.stack(
            [feature.get("labels", feature["input_ids"]) for feature in features]
        )
    ).long()
    return {
        "input_ids": input_ids,
        "attention_mask": labels.ne(-100).long(),
        "labels": labels,
    }


print("Curve validation sequences:", len(curve_eval_dataset))
print("Validation probe SHA-256:", probe_sha256)
print("Experiment fingerprint:", experiment_fingerprint)


## 4. Train one epoch and measure perplexity at fixed steps

The callback measures time spent inside optimizer steps. Validation and SciQ time are excluded from the GPU-hour x-axis, so the x-axis represents training cost.


In [ ]:
wandb.login(key=WANDB_API_KEY)
run = wandb.init(
    project=CONFIG["wandb_project"],
    group=CONFIG["wandb_group"],
    name=f"qwen2.5-0.5b-{VARIANT}-full-efficiency",
    config={
        **CONFIG,
        "variant": VARIANT,
        "training_plan": training_plan,
        "dataset_sha256": variant_info["sha256"],
        "source_manifest_sha256": source_manifest_sha256,
        "probe_sha256": probe_sha256,
        "experiment_fingerprint": experiment_fingerprint,
        "packing": packing,
        "gpu": gpu_name,
    },
)

print("Running a 10-example SciQ smoke test before training")
torch.cuda.reset_peak_memory_stats()
smoke_started = time.perf_counter()
smoke_result = simple_evaluate(
    model="hf",
    model_args={
        "pretrained": CONFIG["model_id"],
        "revision": CONFIG["model_revision"],
        "dtype": "float16",
    },
    tasks=[SCIQ_TASK],
    num_fewshot=0,
    batch_size=CONFIG["sciq_batch_size"],
    device="cuda:0",
    limit=CONFIG["sciq_smoke_examples"],
    bootstrap_iters=100,
    log_samples=False,
    apply_chat_template=False,
    random_seed=CONFIG["seed"],
    numpy_random_seed=CONFIG["seed"],
    torch_random_seed=CONFIG["seed"],
    fewshot_random_seed=CONFIG["seed"],
)
smoke_metrics = extract_sciq_metrics(
    smoke_result,
    variant=VARIANT,
    stage="smoke",
    elapsed_seconds=time.perf_counter() - smoke_started,
    peak_cuda_bytes=torch.cuda.max_memory_allocated(),
)
run.log({
    "smoke/sciq_examples": smoke_metrics["examples"],
    "smoke/sciq_acc_norm": smoke_metrics["acc_norm"],
})
del smoke_result
gc.collect()
torch.cuda.empty_cache()


class TrainingClockCallback(TrainerCallback):
    def __init__(self, variant, tokens_per_update, total_train_tokens):
        self.variant = variant
        self.tokens_per_update = tokens_per_update
        self.total_train_tokens = total_train_tokens
        self.train_seconds = 0.0
        self.step_started = None
        self.validation_by_step = {}

    def on_step_begin(self, args, state, control, **kwargs):
        torch.cuda.synchronize()
        self.step_started = time.perf_counter()

    def on_step_end(self, args, state, control, **kwargs):
        if self.step_started is not None:
            torch.cuda.synchronize()
            self.train_seconds += time.perf_counter() - self.step_started
            self.step_started = None

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        metrics = metrics or {}
        if "eval_loss" not in metrics:
            return
        loss = float(metrics["eval_loss"])
        step = int(state.global_step)
        row = {
            "variant": self.variant,
            "global_step": step,
            "cumulative_train_tokens": tokens_at_step(
                step, self.tokens_per_update, self.total_train_tokens
            ),
            "cumulative_train_gpu_hours": self.train_seconds / 3600.0,
            "validation_loss": loss,
            "validation_perplexity": perplexity_from_loss(loss),
        }
        self.validation_by_step[step] = row
        run.log(
            {
                "curve/global_step": step,
                "curve/cumulative_train_tokens": row["cumulative_train_tokens"],
                "curve/cumulative_train_gpu_hours": row["cumulative_train_gpu_hours"],
                "curve/validation_loss": row["validation_loss"],
                "curve/validation_perplexity": row["validation_perplexity"],
            }
        )


model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_id"],
    revision=CONFIG["model_revision"],
    torch_dtype=torch.float32,
).to("cuda")
model.config.use_cache = False
model.gradient_checkpointing_enable()

clock = TrainingClockCallback(
    VARIANT,
    tokens_per_update=tokens_per_update,
    total_train_tokens=variant_plan["train_input_tokens"],
)
training_args = TrainingArguments(
    output_dir=str(TRAINER_DIR),
    overwrite_output_dir=True,
    num_train_epochs=1.0,
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    max_grad_norm=CONFIG["max_grad_norm"],
    fp16=CONFIG["fp16"],
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=CONFIG["curve_interval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["curve_interval_steps"],
    save_total_limit=None,
    save_only_model=True,
    logging_steps=CONFIG["logging_steps"],
    logging_first_step=True,
    report_to=["wandb"],
    run_name=f"qwen2.5-0.5b-{VARIANT}-full-efficiency",
    seed=CONFIG["seed"],
    data_seed=CONFIG["seed"],
    dataloader_num_workers=2,
    remove_unused_columns=False,
    prediction_loss_only=True,
    optim="adamw_torch",
    save_safetensors=True,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=curve_eval_dataset,
    data_collator=causal_lm_collator,
    processing_class=tokenizer,
    callbacks=[clock],
)

print("Running step-0 validation probe")
trainer.evaluate()
train_output = trainer.train()
if trainer.state.global_step != updates_per_epoch:
    raise RuntimeError(
        f"Expected {updates_per_epoch} optimizer steps, got {trainer.state.global_step}"
    )
print("Running final validation probe")
trainer.evaluate()

train_metrics = dict(train_output.metrics)
train_metrics.update({
    "variant": VARIANT,
    "train_input_tokens": variant_plan["train_input_tokens"],
    "pure_training_gpu_hours": clock.train_seconds / 3600.0,
})
trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(FINAL_MODEL_DIR)
for checkpoint_dir in TRAINER_DIR.glob("checkpoint-*"):
    tokenizer.save_pretrained(checkpoint_dir)

missing_validation_steps = sorted(set(expected_steps) - set(clock.validation_by_step))
if missing_validation_steps:
    raise RuntimeError(f"Missing validation measurements at steps {missing_validation_steps}")
validation_rows = [clock.validation_by_step[step] for step in expected_steps]
print(json.dumps(train_metrics, indent=2))

# Persist the expensive training output before starting the long SciQ sweep.
training_progress = {
    "schema_version": 2,
    "variant": VARIANT,
    "experiment_identity": experiment_identity,
    "experiment_fingerprint": experiment_fingerprint,
    "variant_plan": variant_plan,
    "packing": packing,
    "train_metrics": train_metrics,
    "curve_probe": {
        "probe_sha256": probe_sha256,
        "validation_rows": validation_rows,
    },
}
TRAINING_PROGRESS_PATH.write_text(
    json.dumps(training_progress, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
for local_path in FINAL_MODEL_DIR.rglob("*"):
    if local_path.is_file():
        relative = local_path.relative_to(FINAL_MODEL_DIR).as_posix()
        s3.upload_file(
            str(local_path),
            S3_BUCKET,
            f"{S3_OUTPUT_PREFIX}final/{relative}",
        )
s3.upload_file(
    str(TRAINING_PROGRESS_PATH),
    S3_BUCKET,
    f"{S3_OUTPUT_PREFIX}training-progress.json",
)
run.summary["checkpoint_s3_uri"] = f"s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}final/"
print("Training output safely stored at:", run.summary["checkpoint_s3_uri"])


## 5. Evaluate complete SciQ at every curve point

Step 0 uses the original Qwen Base model. Intermediate checkpoints are local model-only checkpoints. They are not uploaded to S3 after their metrics have been collected.


In [ ]:
checkpoint_paths = {0: CONFIG["model_id"]}
for step in expected_steps[1:]:
    if step == updates_per_epoch:
        checkpoint_paths[step] = str(FINAL_MODEL_DIR)
    else:
        local_path = TRAINER_DIR / f"checkpoint-{step}"
        if not local_path.is_dir():
            raise RuntimeError(f"Missing local checkpoint: {local_path}")
        checkpoint_paths[step] = str(local_path)

del trainer, model
gc.collect()
torch.cuda.empty_cache()

sciq_rows = []
for step in expected_steps:
    checkpoint = checkpoint_paths[step]
    print(f"SciQ: {VARIANT}, step={step}, checkpoint={checkpoint}")
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    model_args = {"pretrained": checkpoint, "dtype": "float16"}
    if step == 0:
        model_args["revision"] = CONFIG["model_revision"]
    harness_result = simple_evaluate(
        model="hf",
        model_args=model_args,
        tasks=[SCIQ_TASK],
        num_fewshot=0,
        batch_size=CONFIG["sciq_batch_size"],
        device="cuda:0",
        limit=CONFIG["sciq_examples"],
        bootstrap_iters=CONFIG["sciq_bootstrap_iters"],
        log_samples=False,
        apply_chat_template=False,
        random_seed=CONFIG["seed"],
        numpy_random_seed=CONFIG["seed"],
        torch_random_seed=CONFIG["seed"],
        fewshot_random_seed=CONFIG["seed"],
    )
    extracted = extract_sciq_metrics(
        harness_result,
        variant=VARIANT,
        stage="full",
        elapsed_seconds=time.perf_counter() - started,
        peak_cuda_bytes=torch.cuda.max_memory_allocated(),
    )
    row = {
        "variant": VARIANT,
        "global_step": step,
        "sciq_acc": extracted["acc"],
        "sciq_acc_norm": extracted["acc_norm"],
        "sciq_acc_stderr": extracted["acc_stderr"],
        "sciq_acc_norm_stderr": extracted["acc_norm_stderr"],
        "sciq_examples": extracted["examples"],
    }
    sciq_rows.append(row)
    SCIQ_PROGRESS_PATH.write_text(
        json.dumps(
            {
                "schema_version": 2,
                "variant": VARIANT,
                "experiment_fingerprint": experiment_fingerprint,
                "completed_steps": [item["global_step"] for item in sciq_rows],
                "rows": sciq_rows,
            },
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )
    s3.upload_file(
        str(SCIQ_PROGRESS_PATH),
        S3_BUCKET,
        f"{S3_OUTPUT_PREFIX}sciq-progress.json",
    )
    run.log({
        "curve/global_step": step,
        "curve/sciq_acc": row["sciq_acc"],
        "curve/sciq_acc_norm": row["sciq_acc_norm"],
    })
    del harness_result
    gc.collect()
    torch.cuda.empty_cache()

curve_rows = merge_curve_measurements(validation_rows, sciq_rows)
write_curve_csv(CURVE_CSV_PATH, curve_rows)
run.log({
    "curve/results": wandb.Table(
        data=[[row[column] for column in CURVE_COLUMNS] for row in curve_rows],
        columns=list(CURVE_COLUMNS),
    )
})
display(curve_rows)


## 6. Full final peS2o validation evaluation

This final check uses the same 320 S2ORC and 680 S2AG documents as the earlier perplexity experiment. It is reported separately because the intermediate curve uses the smaller fixed probe.


In [ ]:
full_records = validation_records
final_model = AutoModelForCausalLM.from_pretrained(
    FINAL_MODEL_DIR,
    torch_dtype=torch.float16,
).to("cuda")
final_model.eval()

full_source_results = {}
for source in ("s2orc", "s2ag"):
    full_source_results[source] = evaluate_source(
        model=final_model,
        packed_sequences=iter_packed_sequences(
            full_records[source], tokenizer, CONFIG["sequence_length"]
        ),
        source=source,
        batch_size=CONFIG["eval_batch_size"],
        device="cuda",
        log_every_steps=25,
    )
    print(
        f"{source}: PPL={full_source_results[source]['perplexity']:.4f}, "
        f"tokens={full_source_results[source]['predicted_tokens']:,}"
    )

full_overall = combine_source_metrics(full_source_results.values())
run.log({
    "final_full_validation/loss": full_overall["loss"],
    "final_full_validation/perplexity": full_overall["perplexity"],
})
print("Full final validation:", json.dumps(full_overall, indent=2))
del final_model, full_records, validation_records
gc.collect()
torch.cuda.empty_cache()


## 7. Save curve results

The final model was uploaded immediately after training. The completed JSON and CSV are now uploaded. Intermediate checkpoints are deliberately kept local to avoid unnecessary S3 storage cost.


In [ ]:
result = {
    "schema_version": 2,
    "variant": VARIANT,
    "experiment_identity": experiment_identity,
    "experiment_fingerprint": experiment_fingerprint,
    "config": CONFIG,
    "training_plan": training_plan,
    "variant_plan": variant_plan,
    "packing": packing,
    "train_metrics": train_metrics,
    "curve_probe": {
        "sequences": CONFIG["curve_validation_sequences"],
        "probe_sha256": probe_sha256,
        "sequence_token_sha256": probe_sequence_hashes,
        "rows": curve_rows,
    },
    "full_final_validation": {
        "sources": full_source_results,
        "overall": full_overall,
    },
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        **environment_compatibility,
        "wandb": wandb.__version__,
    },
    "wandb_run_id": run.id,
    "wandb_run_url": run.url,
}
CURVE_RESULT_PATH.write_text(
    json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

s3.upload_file(
    str(CURVE_RESULT_PATH), S3_BUCKET, f"{S3_OUTPUT_PREFIX}curve-results.json"
)
s3.upload_file(
    str(CURVE_CSV_PATH), S3_BUCKET, f"{S3_OUTPUT_PREFIX}curve-results.csv"
)

artifact = wandb.Artifact(
    name=f"qwen2.5-0.5b-{VARIANT}-full-efficiency-results",
    type="evaluation",
    metadata={
        "variant": VARIANT,
        "train_input_tokens": variant_plan["train_input_tokens"],
        "experiment_fingerprint": experiment_fingerprint,
    },
)
artifact.add_dir(str(RESULTS_DIR))
run.log_artifact(artifact)
run.summary["train_input_tokens"] = variant_plan["train_input_tokens"]
run.summary["experiment_fingerprint"] = experiment_fingerprint
run.summary["pure_training_gpu_hours"] = train_metrics["pure_training_gpu_hours"]
run.summary["final_sciq_acc_norm"] = curve_rows[-1]["sciq_acc_norm"]
run.summary["final_curve_perplexity"] = curve_rows[-1]["validation_perplexity"]
run.summary["final_full_perplexity"] = full_overall["perplexity"]
print("W&B:", run.url)
print("Results:", f"s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}curve-results.json")
wandb.finish()
